# Practice 2: Statistical Analysis of Jazz Solos using Weimar Jazz Database

---

Welcome to Practice 2. 

- In **Practice 1** you worked with **music21** on symbolic scores.

Today we do something different: we work with **tabular data** — rows and columns like a spreadsheet — and apply the **statistical ideas from the lecture** (descriptive statistics, hypothesis tests, correlation). There is **no symbolic music processing** in this notebook: everything is numbers and categories that describe tracks.

We will work with the [Weimar Jazz Database (WJazzD)](https://jazzomat.hfm-weimar.de/index.html)
 — a corpus of **456 transcribed jazz solos** from the history of jazz, from Louis Armstrong (1928) to contemporary postbop. We are not going to work with the melodies directly, but with the features extracted from the melodies. For each solo we have:

1. **A table containing features and metadata** (`Weimars_solos_with_features_and_metadata.csv`) — one row per solo, many numerical columns describing the melody (pitch range, interval entropy, event density, swing ratio, ...). These features were extracted with the **[MeloSpy](https://jazzomat.hfm-weimar.de/download/downloads/MSS_GUI_V_1_4_1.msi)** toolkit, which is specifically designed for jazz melody analysis. It was developed specifically to work with Weimar's Jazz Database, and contains more possibilities than the features shown her. Each file also has metadata, describing the performer, instrument, style, recording year, key, tempo, etc.
2. **The original MIDI files** — the symbolic transcriptions themselves, one `.mid` file per solo.

**What you will do today**

1. **Load** the features and metadata.
2. **Describe** the data with summary statistics and plots
3. Compare two famous players (**Parker vs. Davis**) with a **t-test**
4. Compare **all jazz styles** with a **one-way ANOVA**
5. Ask whether jazz solos have become more complex over time, using **correlation**
6. Test whether **tonal approach** (blues / functional / modal) is associated with **style** using a **chi-squared test**
7. Load a MIDI file with **music21** and visually inspect one of the solos we compared

Each section of the notebook follows the same rhythm: state a **research question**, pick the right **test**, check **assumptions**, compute the **test statistic**, report a **p-value** and an **effect size**, then **interpret**.


## Part 1: Setup (Colab)

Run the cell below once at the start of the session. It installs the Python packages we need.


In [ ]:
# Install packages (Colab / first run). Safe to run again.
%pip install pandas numpy matplotlib seaborn scipy statsmodels music21 --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

RNG_SEED = 42
np.random.seed(RNG_SEED)

BG = "#F7F7F7"
# A palette for the six main jazz styles (plus FREE, which is rare)
STYLE_PALETTE = {
    "TRADITIONAL": "#8B7355",
    "SWING":       "#FFC000",
    "BEBOP":       "#C00000",
    "COOL":        "#AFCDCA",
    "HARDBOP":     "#7F3FBF",
    "POSTBOP":     "#3A3A3A",
    "FREE":        "#E07A5F",
}
plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": "#888888",
    "axes.labelcolor": "#222222",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "grid.color": "#CCCCCC",
    "grid.linestyle": "--",
    "grid.alpha": 0.5,
})

print("Libraries loaded. You're ready to go.")

---
## Part 2: Load and merge the data

Out features are in a CSV file, which is a format storing tables with data. There is a way of loading files in different format into programming language's internal representation. Last time we used music21 to load .midi files. This time we are going to use a library called **pandas** to load a table.

### What is pandas?

**pandas** is a Python library for working with **tables** of data. The main object is a **DataFrame**: think of it as a spreadsheet where each **row** is one track (one observation) and each **column** is one variable (e.g. tempo, genre).

In Practice 1 you saw **lists** and **loops**. Here we mostly call **ready-made functions** on a DataFrame: `df.head()`, `df.describe()`, `df.groupby(...)`, etc.


### Common DataFrame commands (you will see these a lot)

| Command | What it does |
|---------|----------------|
| `df.head()` | Shows the first 5 rows (a quick look at the table). |
| `df.info()` | Lists every column, its **dtype**, and how many **non-null** values there are. |
| `df.describe()` | Summary numbers for numeric columns: count, mean, standard deviation, min, max, quartiles. |
| `df.groupby("col")` | Splits the table by a category (e.g. genre) so you can compute means or counts **per group**. |
| `.round(3)` | Rounds numbers to 3 decimal places so tables are easier to read (not more “accurate”). |





In [3]:
import urllib.request

url = "https://github.com/aljanaki/Digital_musicology/blob/50f011b927b159c756979d5126d04c30a0ff43fa/Practice%20sessions/Practice%202/Weimars_solos_with_features_and_metadata.csv"
urllib.request.urlretrieve(url, "data.csv")

df = pd.read_csv("data.csv")
print("Table:", df.shape)

ParserError: Error tokenizing data. C error: Expected 1 fields in line 38, saw 2


In [ ]:
df.head()

In [ ]:
df.info(verbose=True)

### Reading `df_all.info()`

- **Dtype**: The kind of value stored in the column. **`object`** usually means text; **`int64`** whole numbers; **`float64`** decimals; **`bool`** True/False.

### A note on columns

The features file has many columns (hundreds). We don't need all of them today. The ones we'll use are:

| Feature | Meaning |
|---------|---------|
| `pitch_range` | Highest minus lowest note of the solo, in semitones. |
| `pitch_entropy` | How "varied" the pitch distribution is (higher = more pitch classes used roughly equally). |
| `int_mean` | Mean of signed semitone intervals between consecutive notes (positive = upward tendency). |
| `abs_int_mean` | Mean **absolute** interval size in semitones (ignores direction). |
| `event_density` | Notes per second — how "fast" the player is spitting out notes. |
| `int_entropy` | How varied the interval distribution is. |
| `avgtempo` | Tempo in BPM at which the solo was played. |
| `mean_swing_ratio` | Beat-upbeat ratio: 1.0 = straight eighths, 2.0 = "textbook" swing eighths. |
| `note_count` | Total number of notes in the solo. |

And from the metadata side: `performer`, `style`, `instrument`, `key`, `tonality_type`, `rhythmfeel`, `tempoclass`, `recordingyear`.


In [ ]:
# Quick look at the style distribution — this matters for hypothesis testing
print("Number of solos per style:")
print(df["style"].value_counts())

In [ ]:
# And the performers with the most solos
print("Top 10 performers by solo count:")
print(df["performer"].value_counts().head(10))

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 1 — Explore the data</b><br><br>
In a new cell below, try <code>df.describe()</code> on just a few columns, e.g.:<br><code>df[['pitch_range', 'event_density', 'avgtempo']].describe()</code><br>Also try <code>df['instrument'].value_counts()</code> — which instruments dominate this corpus?
</div>

---
## Part 3: Descriptive statistics and visualisation

Before we test anything, we **look** at the data. Summary statistics and plots are how we decide which tests make sense and which assumptions might be violated.


In [ ]:
# Summary statistics for a handful of musically interesting features
feat_cols = ["pitch_range", "pitch_entropy", "int_mean", "abs_int_mean",
             "event_density", "int_entropy", "avgtempo", "note_count"]
df[feat_cols].describe().round(2)

In [ ]:
# Histogram: event_density (notes per second) across all 456 solos
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["event_density"].dropna(), bins=30, color="#9696E0", edgecolor="white")
ax.set_xlabel("Event density (notes per second)")
ax.set_ylabel("Count of solos")
ax.set_title("Distribution of event density across 456 jazz solos")
ax.grid(True)
plt.tight_layout()
plt.show()

print("Mean event_density:",   round(df["event_density"].mean(),   3))
print("Median event_density:", round(df["event_density"].median(), 3))

### What do we see?

Most solos cluster around **4–7 notes per second**, with a **long right tail** of very fast solos (10+ notes per second — think bebop heads taken at burnout tempo). Because of this tail, the **mean** is slightly higher than the **median**: the distribution is **right-skewed**.

This is typical of rate-like music features — they cannot go below zero, but there is no upper limit, so outliers always stretch the right side.


In [ ]:
# Boxplot: event_density by style
# Order styles historically (matches the chronology of jazz)
style_order = ["TRADITIONAL", "SWING", "BEBOP", "COOL", "HARDBOP", "POSTBOP"]

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=df[df["style"].isin(style_order)],
    x="style", y="event_density", order=style_order,
    hue="style", hue_order=style_order, legend=False,
    palette=[STYLE_PALETTE[s] for s in style_order], ax=ax,
)
ax.set_title("Event density by jazz style (chronological order)")
ax.set_ylabel("Notes per second")
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

### How to read this boxplot

Each box shows the **interquartile range** (25th–75th percentile) for one style; the line inside is the **median**; the whiskers extend to typical extremes; dots are outliers.

Reading left-to-right is reading **jazz history**: Traditional (1920s) → Swing (30s–40s) → Bebop (mid-40s) → Cool (50s) → Hardbop (late 50s) → Postbop (60s–today).

The visual story is clear: **Bebop** and later styles tend to have **higher note density** than pre-war Traditional and Swing. This is a classic musicological claim — "bebop players play more notes" — and it's a great candidate for a statistical test later on.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 2 — Another feature</b><br><br>
Make a boxplot of <code>pitch_range</code> or <code>mean_swing_ratio</code> by style, using the same <code>style_order</code>. What story does each feature tell about the history of jazz?
</div>

---
## Part 4: Two-sample t-test — Parker vs. Davis

### Research question

> **Does Charlie Parker play more notes per second than Miles Davis?**

Charlie Parker (1920–1955) is the defining voice of **bebop**, famous for its breakneck tempos and dense lines. Miles Davis (1926–1991) played bebop early in his career but became a leading figure of **cool jazz** and later **modal jazz**, both of which favour more restrained, spacious phrasing. So musicologically we **expect** Parker to have higher `event_density`. Let's check it with a t-test.

- **H₀ (null):** Mean `event_density` is equal for Parker and Davis.
- **H₁ (alternative):** The means differ.

We will:
1. Check assumptions (**normality** with Shapiro–Wilk, **equal variance** with Levene).
2. Run Welch's t-test if variances differ, Student's t-test otherwise.
3. Report the **p-value** and **Cohen's d** (effect size).


In [ ]:
parker = df.loc[df["performer"] == "Charlie Parker",  "event_density"].dropna()
davis  = df.loc[df["performer"] == "Miles Davis",     "event_density"].dropna()

print(f"n Parker: {len(parker):3d}   mean = {parker.mean():.2f} notes/s")
print(f"n Davis:  {len(davis):3d}   mean = {davis.mean():.2f} notes/s")

In [ ]:
# Assumption checks
w_p, p_shap_p = stats.shapiro(parker)
w_d, p_shap_d = stats.shapiro(davis)
print(f"Shapiro–Wilk Parker: W = {w_p:.3f}, p = {p_shap_p:.3f}")
print(f"Shapiro–Wilk Davis:  W = {w_d:.3f}, p = {p_shap_d:.3f}")

w_lev, p_lev = stats.levene(parker, davis)
print(f"Levene (equal variance):  W = {w_lev:.3f}, p = {p_lev:.3f}")

equal_var = p_lev >= 0.05
print(f"Use equal variances in t-test? {equal_var}")

In [ ]:
# The t-test itself
t_stat, p_val = stats.ttest_ind(parker, davis, equal_var=equal_var)
print(f"t = {t_stat:.3f}")
print(f"p = {p_val:.4g}")

# Cohen's d with pooled standard deviation
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    sx, sy = x.std(ddof=1), y.std(ddof=1)
    pooled = np.sqrt(((nx-1)*sx**2 + (ny-1)*sy**2) / (nx+ny-2))
    return (x.mean() - y.mean()) / pooled

d = cohens_d(parker, davis)
print(f"Cohen's d (Parker – Davis) = {d:+.2f}")
print("(|d| ≈ 0.2 small, 0.5 medium, 0.8 large)")

In [ ]:
# Visualize the two distributions
plot_data = pd.DataFrame({
    "event_density": pd.concat([parker, davis], ignore_index=True),
    "performer": ["Parker"]*len(parker) + ["Davis"]*len(davis),
})

fig, ax = plt.subplots(figsize=(7, 4))
sns.violinplot(
    data=plot_data, x="performer", y="event_density",
    hue="performer", legend=False,
    palette=["#C00000", "#3A3A3A"], ax=ax,
)
ax.set_title("Event density — Parker vs. Davis")
ax.set_ylabel("Notes per second")
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

What type of plot is this? What other types of plots are good for visualizing distributions like this? 

### Interpretation

- **p-value**: If it's well below 0.05, we reject H₀ and conclude the two performers really do differ on average. A p-value of e.g. `0.0002` would mean: *if* Parker and Davis had identical mean event-density, we would see a difference this large only about 2 in 10 000 times by chance.
- **Cohen's d**: This tells us *how big* the difference is in standard-deviation units. For this dataset `d` typically comes out around **+1.5 or larger** — Parker really does play *substantially* more notes per second. That is a **very large** effect.

**Both matter.** A large d without a small p could mean your sample is too tiny; a small d with a tiny p could mean your difference is real but too small to be musically interesting. Always report and interpret both.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 3 — Another pairwise comparison</b><br><br>
Compare <b>John Coltrane</b> and <b>Sonny Rollins</b> on <code>abs_int_mean</code> (mean absolute interval size). Coltrane was famous for his scalar "sheets of sound" approach, Rollins for more angular, leaping lines — do we see that in the data? Run the full pipeline: Shapiro, Levene, t-test, Cohen's d.
</div>

---
## Part 5: One-way ANOVA — tempo across jazz styles

### Research question

> **Does the average performance tempo differ across the six main jazz styles?**

A **t-test** compares **two** groups. When we have **more than two**, we use a **one-way ANOVA** (Analysis of Variance). ANOVA asks a single pooled question:

- **H₀:** All style means are equal.
- **H₁:** At least one style mean is different.

If ANOVA is significant, we follow up with **Tukey's HSD** (Honestly Significant Difference) to see **which pairs** of styles actually differ. This is essential: "some styles differ" is not a satisfying finding — we want to know *which ones*.

**Effect size** for ANOVA: **eta-squared** (η²) = fraction of total variance explained by the grouping.


In [ ]:
# Build one list of values per style (filter out rare FREE and any NaNs)
style_order = ["TRADITIONAL", "SWING", "BEBOP", "COOL", "HARDBOP", "POSTBOP"]
groups = [df.loc[df["style"] == s, "avgtempo"].dropna().values for s in style_order]

for s, g in zip(style_order, groups):
    print(f"{s:12s} n = {len(g):3d}   mean tempo = {g.mean():6.1f}   std = {g.std(ddof=1):5.1f}")

In [ ]:
# Quick assumption check: Levene (equal variances across groups)
w_lev, p_lev = stats.levene(*groups)
print(f"Levene (homogeneity of variance): W = {w_lev:.3f}, p = {p_lev:.3g}")

# The ANOVA itself
F_stat, p_anova = stats.f_oneway(*groups)
print(f"ANOVA: F = {F_stat:.3f}, p = {p_anova:.3g}")

# Eta-squared
all_vals = np.concatenate(groups)
grand_mean = all_vals.mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = ((all_vals - grand_mean)**2).sum()
eta2 = ss_between / ss_total
print(f"Eta-squared = {eta2:.3f}   "
      f"({'small' if eta2 < 0.06 else 'medium' if eta2 < 0.14 else 'large'} effect)")

In [ ]:
# Tukey HSD — which pairs of styles differ?
tmp = df[df["style"].isin(style_order)][["style", "avgtempo"]].dropna()
res = pairwise_tukeyhsd(endog=tmp["avgtempo"], groups=tmp["style"], alpha=0.05)
print(res)

fig = res.plot_simultaneous()
plt.title("Tukey HSD — mean tempo by style")
plt.tight_layout()
plt.show()

### How to read Tukey HSD

Each row of the table is a pair of styles. `meandiff` is the difference of the two group means; `p-adj` is the p-value **adjusted for multiple comparisons**; `reject` is `True` when we can say the two styles really differ.

The **plot** shows each style's 95% confidence interval for the mean. **If two intervals do not overlap**, those styles differ significantly — a quick visual summary of the whole table.

Is **Bebop** faster than **Traditional** and **Swing**? What about Bebop and later styles (Cool, Hardbop, Postbop)?


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 4 — Another ANOVA</b><br><br>
Run a one-way ANOVA for <code>int_entropy</code> (interval entropy — how varied the interval choices are) across the six main styles. Is there a significant difference? Which pairs differ according to Tukey?
</div>

---
## Part 6: Correlation — has jazz become more complex over time?

### Research question

> **Is there a relationship between the year a solo was recorded and its pitch complexity?**

This is one of the classic **historical** questions in computational jazz studies. Previous analyses of the WJazzD have reported that several complexity features tend to increase with recording year — jazz soloists in the postbop era use a wider, more varied pitch vocabulary than in the swing era.

We will test this with **correlation** between `recordingyear` and `pitch_entropy`.

- **Pearson r** measures **linear** relationship.
- **Spearman ρ** measures **monotonic** relationship based on ranks (more robust to outliers).

When they agree you can be more confident the relationship is real. When they differ noticeably, the relationship is likely **nonlinear** or driven by outliers.

**Remember:** Correlation is *not* causation. A correlation with time can reflect many underlying mechanisms — changes in instruments, recording technology, audience expectations, style conventions — not a unified "jazz got more complex" story.


In [ ]:
# Keep only rows where both variables are present
valid = df[["recordingyear", "pitch_entropy"]].dropna()
print(f"Usable solos: {len(valid)}")

r_p, p_p = stats.pearsonr(valid["recordingyear"],  valid["pitch_entropy"])
r_s, p_s = stats.spearmanr(valid["recordingyear"], valid["pitch_entropy"])
print(f"Pearson:  r = {r_p:+.3f}   p = {p_p:.3g}")
print(f"Spearman: ρ = {r_s:+.3f}   p = {p_s:.3g}")

In [ ]:
# Scatter plot with a linear fit
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(valid["recordingyear"], valid["pitch_entropy"],
           alpha=0.4, s=20, color="#2A2A64")
m, b = np.polyfit(valid["recordingyear"], valid["pitch_entropy"], 1)
xs = np.linspace(valid["recordingyear"].min(), valid["recordingyear"].max(), 100)
ax.plot(xs, m*xs + b, color="#DE5B59", linewidth=2,
        label=f"Linear fit (Pearson r = {r_p:+.2f})")
ax.set_xlabel("Recording year")
ax.set_ylabel("Pitch entropy")
ax.set_title("Pitch entropy over the history of jazz (1925–2009)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Interpretation

In this corpus the correlation is typically **weakly positive** — jazz *has* gotten somewhat more pitch-varied over time, but the cloud is wide. The `r` value tells you the strength:

| `r` | Rough interpretation |
|-------|----------------------|
| 0.0 – 0.1 | Essentially no relationship |
| 0.1 – 0.3 | Weak |
| 0.3 – 0.5 | Moderate |
| 0.5 – 0.7 | Strong |
| 0.7 – 1.0 | Very strong |

The **p-value** tells you whether the observed correlation is unlikely under the null hypothesis of no true linear/monotonic relationship. With ~450 data points even weak correlations will be highly significant — so pay attention to the **size** of `r`, not just the p-value.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 5 — A different feature over time</b><br><br>
Repeat this correlation analysis for <code>int_entropy</code> (how varied the interval choices are). Is the trend stronger or weaker than for pitch entropy? Also check <code>ratio_chromatic_sequences</code> — did jazz become more chromatic over time?
</div>

---
## Part 7: Chi-squared — style × tonality type

### Research question

> **Is the distribution of tonality types (functional, blues, modal, free) associated with jazz style?**

Both variables are **categorical**:

- `style`: historical period of the solo (TRADITIONAL, SWING, BEBOP, COOL, HARDBOP, POSTBOP).
- `tonality_type`: the general harmonic approach — `FUNCTIONAL` (standard tonal harmony with ii–V–I progressions), `BLUES` (blues form), `MODAL` (static-scale / modal harmony, think *Kind of Blue* or *A Love Supreme*), `FREE` (free jazz).

Musicologically we **expect** modal solos to cluster in **Postbop** (Davis's *Kind of Blue* from 1959 is sometimes taken as the birth of modal jazz), and blues to be roughly evenly spread across styles.

- **H₀:** Style and tonality type are **independent**.
- **H₁:** They are **associated**.

The test: **Pearson's chi-squared**. Effect size: **Cramér's V**.

**Watch out:** chi-squared becomes unreliable if too many **expected** cell counts are below 5. We will check that.

In [ ]:
# Build the contingency table
ct = pd.crosstab(df["style"], df["tonality_type"])
# Reorder rows chronologically
ct = ct.reindex(["TRADITIONAL", "SWING", "BEBOP", "COOL", "HARDBOP", "POSTBOP"])
print("Observed counts:")
print(ct)

In [ ]:
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f"Chi-squared statistic: {chi2:.3f}")
print(f"Degrees of freedom:    {dof}")
print(f"p-value:               {p_chi:.3g}")
print()
print("Expected counts under H0 (independence):")
print(np.round(expected, 1))
print()
print(f"All expected ≥ 5? {bool(np.all(expected >= 5))}")

# Cramér's V — effect size
n = ct.values.sum()
r, c = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, c) - 1)))
print(f"Cramér's V = {cramers_v:.3f}")
print("(Rule of thumb: 0.1 small, 0.3 medium, 0.5 large)")

In [ ]:
# Heatmap of the observed counts — easier to read than a raw table
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrBr", cbar=True, ax=ax)
ax.set_title("Style × Tonality type (observed counts)")
ax.set_xlabel("Tonality type")
ax.set_ylabel("Style")
plt.tight_layout()
plt.show()

### Interpretation

If `p_chi` is very small and Cramér's V is borderline medium (~0.3), we have some evidence that **style and tonality are associated** — certain styles concentrate certain tonal approaches.

What can you say about harmony use and style based on what you found? 

How is Postbop different from other styles? 
What can we say about blues tonality use over time? 

**Important caveat:** chi-squared tells you *that* there is an association, not *which cells drive it*. To inspect that, look at **residuals** (observed – expected) per cell — positive residuals show where a style has *more* of a tonality than expected under independence.


<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 6 — Style × rhythm feel</b><br><br>
Build <code>pd.crosstab(df['style'], df['rhythmfeel'])</code> and run a chi-squared test. Rhythm feel can be SWING, LATIN, EVEN, etc. Are some rhythmic feels associated with specific historical styles?<br><em>Tip:</em> First look at <code>df['rhythmfeel'].value_counts()</code> — if a category has very few solos, the expected-count rule may be violated.
</div>

---
## Part 8: From numbers back to music — loading a MIDI with music21

Statistical features are powerful but they **abstract away** the music. Let's close the loop by going back to one of the solos we've been talking about and **listening to / looking at** it.

The WJazzD ships MIDIs in a zip archive. We'll download and unzip them, then pick one Parker solo and inspect it with **music21**.


In [ ]:
import urllib.request, zipfile, os

MIDI_URL = "https://jazzomat.hfm-weimar.de/download/downloads/RELEASE2.0_mid_unquant.zip"
ZIP_PATH = "wjazz_midis.zip"
MIDI_DIR = "wjazz_midis"

if not os.path.exists(MIDI_DIR):
    print("Downloading MIDI archive (~4 MB)...")
    urllib.request.urlretrieve(MIDI_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(MIDI_DIR)
    print("Done.")
else:
    print("MIDI folder already exists — skipping download.")

# What's inside? Show a few
midi_files = []
for root, _, files in os.walk(MIDI_DIR):
    for f in files:
        if f.lower().endswith(".mid"):
            midi_files.append(os.path.join(root, f))
print(f"\n{len(midi_files)} MIDI files extracted.")
print("First five:")
for f in midi_files[:5]:
    print(" ", f)

In [ ]:
# Pick one Parker solo to look at
parker_midis = [f for f in midi_files if "CharlieParker" in f or "Parker" in f]
print("Parker MIDI files available (first 5):")
for f in parker_midis[:5]:
    print(" ", os.path.basename(f))

midi_path = parker_midis[0]   # whichever one comes first
print(f"\nLoading: {os.path.basename(midi_path)}")

In [ ]:
from music21 import converter, note

score = converter.parse(midi_path)

# Pull out all note events
notes = [n for n in score.recurse().notes if isinstance(n, note.Note)]

# Pitch range (semitones) — let's compute it ourselves and compare to the CSV
midi_pitches = [n.pitch.midi for n in notes]
pitch_range_m21 = max(midi_pitches) - min(midi_pitches)
print(f"Pitch range (music21):  {pitch_range_m21} semitones")

# Now pull the same from the features table, by matching the filename
solo_id = os.path.basename(midi_path).replace(".mid", "") + ".sv"
row = df[df["id"] == solo_id]
if len(row):
    print(f"Pitch range (MeloSpy): {int(row['pitch_range'].iloc[0])} semitones")
else:
    print(f"(No matching row in the features table for {solo_id}. "
          "Different Parker solo — try another.)")

### What just happened

We loaded a real MIDI transcription of a Charlie Parker solo with music21 (same library as Practice 1), counted the notes, and computed the pitch range **ourselves**. Then we compared our hand-computed number with the value MeloSpy had pre-computed in the CSV.

**They should match** (up to small differences from unquantized vs. quantized representations). This is a good sanity check: the features in the CSV are not magic — they are the same thing you'd compute by hand from the MIDI, just done for you by MeloSpy for all 456 solos at once.

<div style="border: 2px solid #4A90D9; border-radius: 6px; padding: 12px; margin: 10px 0; background-color: #f0f7ff;">
<b>✏️ Task 7 — Your choice</b><br><br>
Pick a <b>different</b> MIDI file (e.g. a Miles Davis or John Coltrane solo) and:<br>1. Load it with <code>music21</code>.<br>2. Display the sheet music.<br>3. Count the notes and compute the pitch range by hand.<br>4. Look up the same solo in <code>df</code> and check your numbers against MeloSpy's <code>note_count</code> and <code>pitch_range</code>.<br>If they disagree, think about why — MeloSpy operates on the unquantized transcription, as do you, so the numbers should be extremely close.
</div>

---
## Part 9: Summary — which test when?

| Question type | Variables | Test used today |
|---------------|-----------|-----------------|
| Compare **two** group means | event_density × performer (Parker vs. Davis) | **t-test** (+ Cohen's d) |
| Compare **3+** group means | avgtempo × style | **One-way ANOVA** + Tukey |
| Linear/monotonic relationship of two numeric variables | pitch_entropy × recordingyear | **Pearson / Spearman** |
| Association of two categorical variables | style × tonality_type | **Chi-squared** + Cramér's V |

**Practical notes to take home**

- With 456 solos, **tiny differences produce tiny p-values** — always look at **effect size** (Cohen's d, η², Cramér's V, r).
- Always **check assumptions** before trusting a test: Shapiro–Wilk for normality, Levene for equal variances.
- **Correlation is not causation.** A historical trend could be driven by many confounded variables (instruments, recording quality, style conventions all changed together).
- **Musicological plausibility matters.** A highly significant result that contradicts everything we know about the music is more likely to reflect a data or analysis issue than a real discovery.

---

### Homework

The homework (Homework 2) will ask you to run a parallel set of analyses on the same dataset — different performers, different features, different questions — and to write a short interpretation of each result. Good luck!
